# TBCA — Text-Based Conjoint Analysis
## Empirical Proof-of-Concept: Amazon Laptop Reviews
**Paper**: *From Utility Theory to Algorithmic Choice* (Anastasiadis & Ampeliotis, 2026)

---
### Pipeline
1. Download dataset from Kaggle
2. Run ABSA (aspect extraction + sentiment scoring)
3. Prospect Theory debiasing
4. MAUT utility estimation (OLS → part-worth weights)
5. Softmax / MNL choice probabilities
6. Axiomatic consistency verification
7. Export tables + charts for paper

**Runtime**: GPU recommended (T4 free tier on Colab is sufficient)  
**Estimated time**: ~15–20 minutes (ABSA model download + inference)

## Cell 1 — Install dependencies
> Run this first. Restart runtime if prompted.

In [ ]:
# Install all required packages
!pip install -q pyabsa kaggle pandas numpy scikit-learn matplotlib seaborn tabulate

# Verify GPU availability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 2 — Kaggle API setup

**Before running this cell:**
1. Go to [kaggle.com](https://www.kaggle.com) → Your Profile → Settings → API → **Create New Token**
2. A file `kaggle.json` will download with your credentials
3. Upload it to Colab using the code below

In [ ]:
import os
from google.colab import files

# Upload your kaggle.json
print('Upload your kaggle.json file:')
uploaded = files.upload()

# Place it in the correct directory
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured.')

## Cell 3 — Download dataset

Dataset: **Amazon Laptop Review** by Ashwini Shet  
URL: https://www.kaggle.com/datasets/ashwinishet/amazon-laptop-review  
~500 reviews with `reviewText` and `overall` (1–5 stars)

In [ ]:
import pandas as pd
import numpy as np

# Download from Kaggle
!kaggle datasets download -d ashwinishet/amazon-laptop-review -p /content/data --unzip

# Load the CSV (filename may vary — list files first)
import glob
csv_files = glob.glob('/content/data/*.csv') + glob.glob('/content/data/*.json')
print('Files found:', csv_files)

# Load (adjust filename if needed)
try:
    df = pd.read_csv(csv_files[0])
except:
    df = pd.read_json(csv_files[0], lines=True)

print(f'\nDataset shape: {df.shape}')
print('Columns:', list(df.columns))
df.head(3)

In [ ]:
# ── Standardise column names ──────────────────────────────────────────────
# Map whatever columns exist to 'text' and 'rating'
# Common variants: 'reviewText'/'review_text'/'body', 'overall'/'rating'/'stars'

col_map = {}
for c in df.columns:
    cl = c.lower()
    if any(k in cl for k in ['review', 'text', 'body', 'comment']):
        col_map[c] = 'text'
    if any(k in cl for k in ['overall', 'rating', 'star', 'score']):
        col_map[c] = 'rating'

df = df.rename(columns=col_map)

# Keep only relevant columns; drop missing
df = df[['text', 'rating']].dropna()
df['text']   = df['text'].astype(str).str.strip()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
df = df[df['text'].str.len() > 20]   # remove very short texts
df = df.reset_index(drop=True)

print(f'Clean dataset: {len(df)} reviews')
print(df['rating'].value_counts().sort_index())

In [ ]:
# ── Sample for manageable ABSA inference time ─────────────────────────────
# 200 reviews is sufficient for a proof-of-concept
# Increase to 500+ for a more robust paper result

N_SAMPLE = 200   # adjust as needed

# Stratified sample: preserve rating distribution
from sklearn.model_selection import train_test_split

if len(df) > N_SAMPLE:
    df_sample, _ = train_test_split(
        df, train_size=N_SAMPLE,
        stratify=df['rating'].clip(1, 5).astype(int),
        random_state=42
    )
    df_sample = df_sample.reset_index(drop=True)
else:
    df_sample = df.copy()

print(f'Working with {len(df_sample)} reviews')
print(df_sample['rating'].value_counts().sort_index())

## Cell 4 — ABSA: Aspect extraction + sentiment scoring

Using **PyABSA** with the pretrained multilingual ATEPC checkpoint.  
This model:
- Extracts aspect terms (e.g. "battery", "screen", "performance") automatically
- Assigns sentiment polarity: Positive / Negative / Neutral
- Provides a confidence score

**Theoretical link (paper Section 2.3):** Extracted aspects = MAUT attribute dimensions $x_i$.  
Sentiment scores = empirical surrogates for part-worth utilities $u_i(x_i)$.

In [ ]:
from pyabsa import ATEPCCheckpointManager

# Load pretrained ABSA model (downloads ~400MB on first run)
aspect_extractor = ATEPCCheckpointManager.get_aspect_extractor(
    checkpoint='english',
    auto_device=True   # uses GPU if available
)
print('ABSA model loaded.')

In [ ]:
# ── Run ABSA inference ────────────────────────────────────────────────────
# Process in batches to avoid memory issues

BATCH_SIZE = 32
all_results = []

texts = df_sample['text'].tolist()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    result = aspect_extractor.extract_aspect(
        inference_source=batch,
        pred_sentiment=True
    )
    all_results.extend(result)
    print(f'Processed {min(i+BATCH_SIZE, len(texts))}/{len(texts)} reviews', end='\r')

print(f'\nABSA complete. {len(all_results)} results.')

In [ ]:
# ── Parse ABSA output into structured format ──────────────────────────────

records = []
for idx, result in enumerate(all_results):
    aspects   = result.get('aspect',     [])
    sentiments = result.get('sentiment',  [])
    conf      = result.get('confidence', [])

    if not aspects:
        continue

    for j, aspect in enumerate(aspects):
        sent = sentiments[j] if j < len(sentiments) else 'Neutral'
        c    = float(conf[j]) if j < len(conf) else 0.5

        # Convert polarity label to numeric score ∈ [-1, +1]
        if sent == 'Positive':
            score = c
        elif sent == 'Negative':
            score = -c
        else:
            score = 0.0

        records.append({
            'review_idx': idx,
            'rating':     df_sample.loc[idx, 'rating'],
            'aspect':     aspect.lower().strip(),
            'sentiment':  sent,
            'confidence': c,
            'score':      score
        })

df_absa = pd.DataFrame(records)
print(f'Total aspect mentions: {len(df_absa)}')
print(f'Unique aspects found: {df_absa["aspect"].nunique()}')
print('\nTop 20 aspects by frequency:')
print(df_absa['aspect'].value_counts().head(20))

In [ ]:
# ── Map free-form aspects to canonical MAUT attribute dimensions ───────────
# This is the TBCA attribute consolidation step (Section 8.3, Stage 2)

ATTRIBUTE_MAP = {
    # battery_life
    'battery': 'battery_life', 'battery life': 'battery_life',
    'battery performance': 'battery_life', 'charge': 'battery_life',
    'charging': 'battery_life',

    # display_quality
    'screen': 'display_quality', 'display': 'display_quality',
    'resolution': 'display_quality', 'brightness': 'display_quality',
    'monitor': 'display_quality', 'image quality': 'display_quality',

    # performance
    'performance': 'performance', 'speed': 'performance',
    'processor': 'performance', 'cpu': 'performance',
    'gpu': 'performance', 'ram': 'performance',
    'memory': 'performance', 'processing': 'performance',

    # build_quality
    'build': 'build_quality', 'build quality': 'build_quality',
    'design': 'build_quality', 'keyboard': 'build_quality',
    'trackpad': 'build_quality', 'touchpad': 'build_quality',
    'quality': 'build_quality', 'feel': 'build_quality',
    'material': 'build_quality', 'durability': 'build_quality',

    # value_for_money
    'price': 'value_for_money', 'value': 'value_for_money',
    'cost': 'value_for_money', 'worth': 'value_for_money',
    'money': 'value_for_money', 'deal': 'value_for_money',
}

ATTRIBUTES = ['battery_life', 'display_quality', 'performance',
              'build_quality', 'value_for_money']

def map_aspect(aspect_str):
    a = aspect_str.lower().strip()
    # Exact match first
    if a in ATTRIBUTE_MAP:
        return ATTRIBUTE_MAP[a]
    # Partial match
    for key, val in ATTRIBUTE_MAP.items():
        if key in a or a in key:
            return val
    return None   # unmapped aspects are dropped

df_absa['attribute'] = df_absa['aspect'].apply(map_aspect)
df_absa_mapped = df_absa.dropna(subset=['attribute'])

coverage = len(df_absa_mapped) / len(df_absa) * 100
print(f'Mapped aspects: {len(df_absa_mapped)} / {len(df_absa)} ({coverage:.1f}%)')
print('\nAttribute frequency after mapping:')
print(df_absa_mapped['attribute'].value_counts())

In [ ]:
# ── Build feature matrix: one row per review, one column per attribute ─────
# Multiple mentions of the same attribute in one review → averaged

# Pivot: mean sentiment score per review × attribute
pivot = df_absa_mapped.groupby(['review_idx', 'attribute'])['score'].mean().unstack(fill_value=0)

# Ensure all 5 attribute columns exist
for attr in ATTRIBUTES:
    if attr not in pivot.columns:
        pivot[attr] = 0.0
pivot = pivot[ATTRIBUTES]   # enforce column order

# Merge with ratings
ratings_series = df_sample['rating'].rename('rating')
df_features = pivot.join(ratings_series, how='inner')
df_features = df_features.dropna()

print(f'Feature matrix shape: {df_features.shape}')
print(f'Reviews with ≥1 mapped aspect: {len(df_features)}')
df_features.head()

## Cell 5 — Prospect Theory debiasing (Section 7.2)

Parameters: λ = 2.25, α = 0.88 (Tversky & Kahneman, 1992)

Corrects for:
- **Loss aversion bias**: negative reviews over-represented in UGC
- **Diminishing sensitivity**: extreme scores compressed toward center

In [ ]:
LAMBDA = 2.25   # Tversky & Kahneman (1992)
ALPHA  = 0.88

def pt_debias(s):
    """Inverse PT correction: s ∈ [-1,+1] → debiased score."""
    if s == 0:
        return 0.0
    elif s > 0:
        return s ** ALPHA
    else:
        return -(abs(s) ** ALPHA) / LAMBDA

# Apply to each attribute column
X_raw    = df_features[ATTRIBUTES].values.copy()
X_debias = np.vectorize(pt_debias)(X_raw)

y = ((df_features['rating'].values - 1) / 4.0)   # normalise to [0,1]

print('Raw feature matrix (first 3 rows):')
print(pd.DataFrame(X_raw[:3], columns=ATTRIBUTES).round(3))
print('\nDebiased feature matrix (first 3 rows):')
print(pd.DataFrame(X_debias[:3], columns=ATTRIBUTES).round(3))

## Cell 6 — MAUT utility estimation (Section 6.3)

**Û(x) = Σᵢ ŵᵢ · ûᵢ(xᵢ)** (Eq. 10 — MAUT additive form)  
**= W · φ(x) + b** (Eq. 15 — neural linear layer)

We estimate weights via Ridge regression (OLS with L2 regularisation).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# ── OLS / Ridge regression ────────────────────────────────────────────────
ridge_raw    = Ridge(alpha=0.01, fit_intercept=True)
ridge_debias = Ridge(alpha=0.01, fit_intercept=True)

ridge_raw.fit(X_raw,    y)
ridge_debias.fit(X_debias, y)

# R² (5-fold cross-validation)
r2_raw    = cross_val_score(ridge_raw,    X_raw,    y, cv=5, scoring='r2').mean()
r2_debias = cross_val_score(ridge_debias, X_debias, y, cv=5, scoring='r2').mean()

print(f'R² (raw features,    CV-5): {r2_raw:.4f}')
print(f'R² (debiased features, CV-5): {r2_debias:.4f}')

# ── Normalise to MAUT part-worth weights (Σwᵢ=1, wᵢ>0) ──────────────────
w_raw    = ridge_raw.coef_
w_debias = ridge_debias.coef_

w_raw_norm    = np.abs(w_raw)    / np.abs(w_raw).sum()
w_debias_norm = np.abs(w_debias) / np.abs(w_debias).sum()

# ── Summary table ─────────────────────────────────────────────────────────
from tabulate import tabulate

table = []
for a, wr, wd in zip(ATTRIBUTES, w_raw_norm, w_debias_norm):
    delta = wd - wr
    direction = '↑' if delta > 0 else '↓'
    table.append([a, f'{wr:.4f}', f'{wd:.4f}', f'{delta:+.4f} {direction}'])

print('\n── Part-Worth Utility Weights ŵᵢ ──')
print(tabulate(table,
               headers=['Attribute', 'Raw w̃ᵢ', 'Debiased ŵᵢ', 'Δ'],
               tablefmt='github'))
print(f'\nΣ raw:     {w_raw_norm.sum():.6f}')
print(f'Σ debiased: {w_debias_norm.sum():.6f}')

## Cell 7 — Softmax / MNL choice probabilities (Section 6.1)

**Proposition 6.1**: softmax(V)ᵢ = Pᴹᴺᴸᵢ for all i  
Three hypothetical laptops evaluated against the recovered attribute weights.

In [ ]:
# Three hypothetical laptop profiles (attribute levels ∈ [0,1])
ALTERNATIVES = {
    'Laptop A (Premium)':     [0.9, 0.85, 0.80, 0.90, 0.20],
    'Laptop B (Performance)': [0.4, 0.60, 0.95, 0.50, 0.70],
    'Laptop C (Budget)':      [0.7, 0.40, 0.50, 0.30, 0.95],
}

def softmax(v):
    ev = np.exp(v - v.max())
    return ev / ev.sum()

profiles = np.array(list(ALTERNATIVES.values()))
V = profiles @ w_debias_norm   # V_j = Σᵢ ŵᵢ · x_ij
P = softmax(V)

print('── Softmax Choice Probabilities ──')
choice_table = [
    [name, f'{v:.4f}', f'{p:.4f}', f'{p*100:.1f}%']
    for (name, _), v, p in zip(ALTERNATIVES.items(), V, P)
]
print(tabulate(choice_table,
               headers=['Alternative', 'V̂(x)', 'P(choice)', '%'],
               tablefmt='github'))
print(f'\nΣ P(choice) = {P.sum():.6f}  ✓')

## Cell 8 — Axiomatic consistency verification

In [ ]:
checks = {
    'Σ P(choice) = 1   [RUM probability axiom]':      abs(P.sum() - 1) < 1e-9,
    'Σ ŵᵢ = 1          [MAUT normalisation]':          abs(w_debias_norm.sum() - 1) < 1e-9,
    'All Pⱼ ∈ (0,1)    [RUM interior requirement]':    all(0 < p < 1 for p in P),
    'All ŵᵢ > 0        [MAUT monotonicity]':           all(w > 0 for w in w_debias_norm),
}

print('── Axiomatic Consistency Checks ──')
all_pass = True
for check, passed in checks.items():
    status = 'PASS ✓' if passed else 'FAIL ✗'
    print(f'  {status}  {check}')
    if not passed:
        all_pass = False

print()
if all_pass:
    print('All checks PASSED — TBCA output is axiomatically consistent')
    print('with RUM (McFadden, 1974) and MAUT (Keeney & Raiffa, 1993).')
    print('Training on this data is theoretically equivalent to estimating')
    print('an axiomatically consistent utility model (Propositions 6.1–6.2).')
else:
    print('WARNING: Some checks failed — review the pipeline.')

## Cell 9 — Charts for paper

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('TBCA Empirical Results — Amazon Laptop Reviews',
             fontsize=14, fontweight='bold', y=1.02)

# ── Plot 1: Part-worth weights comparison ─────────────────────────────────
ax1 = axes[0]
x = np.arange(len(ATTRIBUTES))
w = 0.35
bars_raw    = ax1.bar(x - w/2, w_raw_norm,    w, label='Raw w̃ᵢ',      color='#5B8DB8', alpha=0.85)
bars_debias = ax1.bar(x + w/2, w_debias_norm, w, label='Debiased ŵᵢ', color='#2E8B57', alpha=0.85)

ax1.set_xticks(x)
short_labels = ['Battery', 'Display', 'Perf.', 'Build', 'Value']
ax1.set_xticklabels(short_labels, fontsize=10)
ax1.set_ylabel('Part-worth weight')
ax1.set_title('Part-worth weights ŵᵢ\n(MAUT, Eq. 10)')
ax1.legend(fontsize=9)
ax1.set_ylim(0, max(w_raw_norm.max(), w_debias_norm.max()) * 1.25)

# ── Plot 2: PT debiasing delta ─────────────────────────────────────────────
ax2 = axes[1]
deltas = w_debias_norm - w_raw_norm
colors = ['#2E8B57' if d > 0 else '#C0392B' for d in deltas]
ax2.bar(short_labels, deltas, color=colors, alpha=0.85)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('PT debiasing effect Δ\n(Eq. 2 correction)')
ax2.set_ylabel('Δ = ŵᵢ − w̃ᵢ')
up_patch   = mpatches.Patch(color='#2E8B57', label='Upward correction')
down_patch = mpatches.Patch(color='#C0392B', label='Downward correction')
ax2.legend(handles=[up_patch, down_patch], fontsize=9)

# ── Plot 3: Softmax choice probabilities ──────────────────────────────────
ax3 = axes[2]
alt_labels = ['Laptop A\n(Premium)', 'Laptop B\n(Performance)', 'Laptop C\n(Budget)']
bar_colors = ['#185FA5', '#1D9E75', '#888780']
bars = ax3.bar(alt_labels, P * 100, color=bar_colors, alpha=0.85)

for bar, pval in zip(bars, P):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{pval*100:.1f}%',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

ax3.set_ylabel('Choice probability (%)')
ax3.set_title('MNL/Softmax probabilities\n(Eq. 12, Prop. 6.1)')
ax3.set_ylim(0, max(P * 100) * 1.2)
ax3.axhline(100/3, color='gray', linestyle='--', linewidth=0.8,
            label='Random baseline (33.3%)')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/content/tbca_results.pdf', dpi=300, bbox_inches='tight')
plt.savefig('/content/tbca_results.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figures saved: tbca_results.pdf / .png')

## Cell 10 — Export results for paper tables

In [ ]:
import json
from google.colab import files

# ── LaTeX table: part-worth weights ──────────────────────────────────────
latex_pw = """\\begin{table}[ht]
\\centering
\\caption{Part-Worth Utility Weights: Raw vs.\\ PT-Debiased (Real Data)}
\\label{tab:part_worths_empirical}
\\begin{tabular}{lrrr}
\\toprule
Attribute & Raw $\\tilde{w}_i$ & Debiased $\\hat{w}_i$ & $\\Delta$ \\\\
\\midrule
"""
for a, wr, wd in zip(ATTRIBUTES, w_raw_norm, w_debias_norm):
    delta = wd - wr
    arrow = '\\uparrow' if delta > 0 else '\\downarrow'
    latex_pw += f"{a.replace('_', ' ').title()} & {wr:.4f} & {wd:.4f} & ${delta:+.4f}\\;{arrow}$ \\\\\n"
latex_pw += f"""\\midrule
$\\Sigma$ & {w_raw_norm.sum():.4f} & {w_debias_norm.sum():.4f} & \\\\
\\midrule
\\multicolumn{{4}}{{l}}{{R\\textsuperscript{{2}} (CV-5): {r2_debias:.4f} (debiased),
{r2_raw:.4f} (raw)}}\\\\
\\bottomrule
\\end{{tabular}}
\\end{{table}}"""

# ── LaTeX table: choice probabilities ────────────────────────────────────
latex_choice = """\\begin{table}[ht]
\\centering
\\caption{Softmax Choice Probabilities (MNL/RUM, Eq.~12)}
\\label{tab:choice_probs_empirical}
\\begin{tabular}{lrr}
\\toprule
Alternative & $\\hat{V}_j$ & $P_j$ \\\\
\\midrule
"""
for (name, _), v, p in zip(ALTERNATIVES.items(), V, P):
    latex_choice += f"{name} & {v:.4f} & {p:.4f} \\\\\n"
latex_choice += f"""\\midrule
$\\Sigma$ & & {P.sum():.4f} \\\\
\\bottomrule
\\end{{tabular}}
\\end{{table}}"""

# Save
with open('/content/tbca_tables.tex', 'w') as f:
    f.write(latex_pw + '\n\n' + latex_choice)

# JSON results
results = {
    'n_reviews': len(df_sample),
    'n_with_aspects': len(df_features),
    'r2_raw': round(r2_raw, 4),
    'r2_debiased': round(r2_debias, 4),
    'part_worths': {
        a: {'raw': round(float(wr), 4), 'debiased': round(float(wd), 4)}
        for a, wr, wd in zip(ATTRIBUTES, w_raw_norm, w_debias_norm)
    },
    'choice_probabilities': {
        name: {'utility': round(float(v), 4), 'prob': round(float(p), 4)}
        for (name, _), v, p in zip(ALTERNATIVES.items(), V, P)
    }
}
with open('/content/tbca_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Download all output files
for fname in ['tbca_tables.tex', 'tbca_results.json',
              'tbca_results.pdf', 'tbca_results.png']:
    try:
        files.download(f'/content/{fname}')
    except:
        print(f'Could not auto-download {fname} — find it in /content/')

print('\nAll outputs ready.')
print('tbca_tables.tex  → paste directly into paper LaTeX source')
print('tbca_results.pdf → Figure for paper')
print('tbca_results.json → structured results')